# 迈克尔逊干涉仪测量光波波长 (逐差法)

本 Notebook 用于通过**迈克尔逊干涉仪（Michelson Interferometer）**测量单色光的波长 $\lambda$。实验记录反射镜 $M_1$ 移动时干涉圆环“涌出”或“陷入”的条纹数与对应位置读数，利用**逐差法**（10 个原始数据分为前后两组逐差）消除初始零点误差与线性漂移，计算波长并完成严格的不确定度传递评定与规范修约。

> **仪器读数说明**：
> * 测微手轮最小分度值为 $0.0001\,\mathrm{mm}$（即 $0.1\,\mu\mathrm{m}$）。
> * 读数时需向下估读一位至 $0.00001\,\mathrm{mm}$（即小数点后第 5 位，对应 $0.01\,\mu\mathrm{m}$ 或 $10\,\mathrm{nm}$）。
> * 仪器误差限 $\Delta_{\mathrm{inst}}$ 通常取最小刻度 $0.0001\,\mathrm{mm}$（或视实验要求取半刻度 $0.00005\,\mathrm{mm}$）。

---
### 实验原理与数学公式

#### 1. 光波波长测量原理与逐差法公式
当动镜 $M_1$ 移动距离 $\Delta d$ 时，中心涌出（或陷入）干涉圆环数 $N$ 满足光程差改变公式：
$$\Delta d = N \cdot \frac{\lambda}{2} \implies \lambda = \frac{2 \Delta d}{N}$$
实验中每“涌出”（或“陷入”）$\Delta n = 50$ 个条纹，记录一次 $M_1$ 的位置读数，共测量 10 个数据 $d_0, d_1, \dots, d_9$（分别对应环数 $0, 50, 100, \dots, 450$）。
采用 5 组逐差法（前后 5 组跨度逐差，逐差跨度环数 $\Delta N = 5 \times 50 = 250$）：
$$\Delta d_i = d_{i+5} - d_i \quad (i = 0, 1, 2, 3, 4)$$
逐差位移平均值：
$$\overline{\Delta d} = \frac{1}{5} \sum_{i=0}^4 \Delta d_i$$
各组对应的单色光波长分量及平均波长：
$$\lambda_i = \frac{2 \Delta d_i}{\Delta N} = \frac{\Delta d_i}{125}, \quad \bar{\lambda} = \frac{2 \overline{\Delta d}}{\Delta N} = \frac{\overline{\Delta d}}{125}$$

#### 2. 不确定度评定公式
* **A 类不确定度**（由逐差项的随机离散性评定）：
  $$s(\Delta d) = \sqrt{\frac{1}{M-1} \sum_{i=0}^{M-1} (\Delta d_i - \overline{\Delta d})^2} \quad (M = 5)$$
  $$u_A(\overline{\Delta d}) = \frac{s(\Delta d)}{\sqrt{M}} = \frac{s(\Delta d)}{\sqrt{5}}$$
  $$u_A(\lambda) = \frac{2 u_A(\overline{\Delta d})}{\Delta N}$$
* **B 类不确定度**（由干涉仪微调测微鼓轮仪器误差限 $\Delta_{\mathrm{inst}}$ 传递）：
  单次读数 $d$ 的仪器标准不确定度为 $u(d) = \frac{\Delta_{\mathrm{inst}}}{\sqrt{3}}$。
  逐差平均值 $\overline{\Delta d} = \frac{1}{M} \sum_{i=0}^{M-1} (d_{i+M} - d_i)$ 共由 $2M$ 个独立读数线性组合而成：
  $$u_B(\overline{\Delta d}) = \sqrt{\frac{2}{M}} u(d) = \sqrt{\frac{2}{5}} \frac{\Delta_{\mathrm{inst}}}{\sqrt{3}}$$
  $$u_B(\lambda) = \frac{2 u_B(\overline{\Delta d})}{\Delta N}$$
* **合成标准不确定度与相对不确定度**：
  $$u_c(\lambda) = \sqrt{u_A^2(\lambda) + u_B^2(\lambda)}, \quad u_r(\lambda) = \frac{u_c(\lambda)}{\bar{\lambda}} \times 100\%$$
* **科学修约**：利用 `scientific_round` 按照国家计量规范（“四舍六入五凑偶”规则），不确定度保留 1 位有效数字，测量结果末位对齐。

#### 3. 钠光 D 双线波长差（选做模块）
若使用钠光灯并观察干涉条纹周期性“视场变亮/变暗模糊”，相邻两次条纹最不清晰（或最清晰）位置动镜移动距离 $\Delta d$ 满足：
$$\Delta \lambda = \lambda_1 - \lambda_2 \approx \frac{\bar{\lambda}^2}{2 \Delta d}$$


In [1]:
import math
from decimal import Decimal
from python.utils import scientific_round

print("迈克尔逊干涉仪计算模块加载完成。")

迈克尔逊干涉仪计算模块加载完成。


### 1. 实验参数与 10 个原始位置读数输入 (表 5-8-1)
> **说明与读数规范**：
> * 仪器刻度为 $0.0001\,\mathrm{mm}$，**加一位估读位后数据保留到小数点后 5 位**（单位：mm，例如 `10.12502`）。
> * `d_vals`：记录干涉环每涌出（或陷入）50 环时的 $M_1$ 位置读数，共 10 个原始数据。
> * 默认提供 He-Ne 激光（$\lambda_0 = 632.8\,\mathrm{nm}$）带有 5 位小数估读的典型实验数据。
> * 若测量钠黄光，可将 `WAVELENGTH_THEORY_NM` 改为 `Decimal("589.3")`，`SOURCE_NAME` 改为 `"钠光"`。

In [2]:
# --- 光源及理论波长 (用于对比相对误差) ---
SOURCE_NAME = "He-Ne 激光"               # 光源名称 (如 "He-Ne 激光" 或 "钠光")
WAVELENGTH_THEORY_NM = Decimal("632.8")  # 理论波长 λ_0 (nm)

# --- 仪器参数 (最小刻度 0.0001 mm，含一位估读) ---
# 仪器误差限 Δ_inst (mm)：通常取最小分度值 0.0001 mm，如果有估读则取半刻度 0.00005 mm
delta_inst = Decimal("0.00005")

# --- 测量条纹计数步长 ---
RING_STEP = 50  # 每组涌出或陷入的条纹个数 (个)

# --- 表 5-8-1: 10 个原始测量位置读数 d_i (mm，保留 5 位小数，含估读位) ---
# 环数对应: 0, 50, 100, 150, 200, 250, 300, 350, 400, 450 环
# 示例测量数据 (可直接替换为您在实验室记录的数据):
d_vals = [
    Decimal("10.17246"), Decimal("10.18890"), Decimal("10.20550"), Decimal("10.22155"), Decimal("10.23710"),  # d_0 ~ d_4
    Decimal("10.25311"), Decimal("10.26912"), Decimal("10.28508"), Decimal("10.30166"), Decimal("10.31762")   # d_5 ~ d_9
]

print(f"当前光源: {SOURCE_NAME}")
print(f"参考理论波长: {WAVELENGTH_THEORY_NM} nm")
print(f"仪器误差限 Δ_inst: {delta_inst} mm (刻度 0.0001 mm，估读至 0.00001 mm)")
print(f"已录入位置读数个数: {len(d_vals)} (需保证为 10 个数据)")

当前光源: He-Ne 激光
参考理论波长: 632.8 nm
仪器误差限 Δ_inst: 0.00005 mm (刻度 0.0001 mm，估读至 0.00001 mm)
已录入位置读数个数: 10 (需保证为 10 个数据)


### 2. 逐差法处理、波长求解与不确定度评定

In [3]:
assert len(d_vals) == 10, f"数据个数必须为 10 个，当前为 {len(d_vals)} 个！"

M = 5  # 逐差对数
delta_N = M * RING_STEP  # 逐差环数间隔: 5 * 50 = 250 个环

deltas = []      # 逐差位移 Δd_i (mm)
lambdas_nm = []  # 各组计算出的波长 λ_i (nm)

print("=" * 68)
print("             表 5-8-1 逐 差 法 数 据 处 理 详 细 表             ")
print("=" * 68)
col_titles = f"{'组号 i':^6}|{'d_i (mm)':^13}|{'d_{i+5} (mm)':^15}|{'Δd_i (mm)':^13}|{'λ_i (nm)':^12}"
print(col_titles)
print("-" * 68)

for i in range(M):
    d_low = d_vals[i]
    d_high = d_vals[i + M]
    diff = d_high - d_low
    # λ = 2 * Δd / ΔN (mm -> nm 乘 10^6)
    lam_nm = (Decimal("2") * diff / Decimal(str(delta_N))) * Decimal("1000000")
    deltas.append(diff)
    lambdas_nm.append(lam_nm)
    print(f"{i:^8}|{str(d_low):^13}|{str(d_high):^15}|  {diff:.5f}  |{lam_nm:.2f}")

print("-" * 68)

# 1. 逐差平均位移与平均波长
mean_delta = sum(deltas) / Decimal(str(M))
mean_lambda_nm = (Decimal("2") * mean_delta / Decimal(str(delta_N))) * Decimal("1000000")

# 2. A 类不确定度评定 (针对 Δd 项)
variance_delta = sum((d - mean_delta)**2 for d in deltas) / Decimal(str(M - 1))
s_delta = Decimal(str(math.sqrt(float(variance_delta))))
u_a_delta = s_delta / Decimal(str(math.sqrt(M)))
u_a_lambda_nm = (Decimal("2") * u_a_delta / Decimal(str(delta_N))) * Decimal("1000000")

# 3. B 类不确定度评定
# 单次测量的标准不确定度 u(d) = Δ_inst / √3
u_d = delta_inst / Decimal(str(math.sqrt(3)))
# 逐差平均值是由 2M 个独立测量值的线性组合，其 B 类不确定度为: √(2/M) * u(d)
u_b_delta = Decimal(str(math.sqrt(2.0 / M))) * u_d
u_b_lambda_nm = (Decimal("2") * u_b_delta / Decimal(str(delta_N))) * Decimal("1000000")

# 4. 合成标准不确定度与相对不确定度
u_c_lambda_nm = Decimal(str(math.sqrt(float(u_a_lambda_nm**2 + u_b_lambda_nm**2))))
u_r_lambda = (u_c_lambda_nm / mean_lambda_nm) * Decimal("100")

# 5. 科学规范修约
lambda_final, u_final = scientific_round(mean_lambda_nm, u_c_lambda_nm)

# 6. 与理论波长相对误差
rel_error = (abs(mean_lambda_nm - WAVELENGTH_THEORY_NM) / WAVELENGTH_THEORY_NM) * Decimal("100")

print("\n" + "=" * 52)
print("        迈 克 尔 逊 干 涉 仪 测 光 波 波 长 结 果        ")
print("=" * 52)
print(f"逐差平均位移 Δd_avg     : {mean_delta:.5f} mm")
print(f"逐差位移实验标准差 s(Δd) : {s_delta:.5f} mm")
print(f"逐差位移 A 类不确定度    : {u_a_delta:.5f} mm")
print(f"逐差位移 B 类不确定度    : {u_b_delta:.5f} mm")
print("-" * 52)
print(f"测量光波波长平均值 λ     : {mean_lambda_nm:.3f} nm")
print(f"A 类不确定度 u_A(λ)      : {u_a_lambda_nm:.3f} nm")
print(f"B 类不确定度 u_B(λ)      : {u_b_lambda_nm:.3f} nm")
print(f"合成标准不确定度 u_c(λ)  : {u_c_lambda_nm:.3f} nm")
print(f"相对不确定度 u_r(λ)      : {u_r_lambda:.2f}%")
print(f"理论参考波长 λ_0         : {WAVELENGTH_THEORY_NM} nm")
print(f"与理论值相对误差 E_r     : {rel_error:.2f}%")
print("-" * 52)
print(f"★ 最终规范修约结果      : λ = {lambda_final} ± {u_final} nm")
print("=" * 52)

             表 5-8-1 逐 差 法 数 据 处 理 详 细 表             
 组号 i |  d_i (mm)   | d_{i+5} (mm)  |  Δd_i (mm)  |  λ_i (nm)  
--------------------------------------------------------------------
   0    |  10.17246   |   10.25311    |  0.08065  |645.20
   1    |  10.18890   |   10.26912    |  0.08022  |641.76
   2    |  10.20550   |   10.28508    |  0.07958  |636.64
   3    |  10.22155   |   10.30166    |  0.08011  |640.88
   4    |  10.23710   |   10.31762    |  0.08052  |644.16
--------------------------------------------------------------------

        迈 克 尔 逊 干 涉 仪 测 光 波 波 长 结 果        
逐差平均位移 Δd_avg     : 0.08022 mm
逐差位移实验标准差 s(Δd) : 0.00042 mm
逐差位移 A 类不确定度    : 0.00019 mm
逐差位移 B 类不确定度    : 0.00002 mm
----------------------------------------------------
测量光波波长平均值 λ     : 641.728 nm
A 类不确定度 u_A(λ)      : 1.493 nm
B 类不确定度 u_B(λ)      : 0.146 nm
合成标准不确定度 u_c(λ)  : 1.500 nm
相对不确定度 u_r(λ)      : 0.23%
理论参考波长 λ_0         : 632.8 nm
与理论值相对误差 E_r     : 1.41%
-------------------------------------

### 3. 选做实验：测定钠光 D 双线的波长差 $\Delta \lambda$
> **原理说明**：
> 钠光包含两条波长极其接近的谱线（$\lambda_1 = 589.592\,\mathrm{nm}, \lambda_2 = 588.995\,\mathrm{nm}$，平均波长 $\bar{\lambda} \approx 589.3\,\mathrm{nm}$）。
> 旋转测微手轮时，两束光干涉条纹的明暗位置发生周期性错开与重合，视场内条纹清晰度呈现“清晰 $\to$ 模糊（最差） $\to$ 清晰”的周期性交替。
> 记录连续条纹最模糊时的动镜位置读数 $d'_k$（保留 5 位小数），相邻两次条纹最模糊之间的动镜移动量为 $\Delta d$，则波长差为：
> $$\Delta \lambda = \lambda_1 - \lambda_2 \approx \frac{\bar{\lambda}^2}{2 \Delta d}$$
> 若记录了多次（如 5 次）条纹最不清晰位置，同样可通过连续差值平均得到 $\Delta d_{\mathrm{avg}}$。

In [ ]:
# --- 钠黄光平均理论波长 λ_bar (nm) ---
LAMBDA_BAR_NM = Decimal("589.3")
DELTA_LAMBDA_THEORY_NM = Decimal("0.60")  # 理论双线波长差 Δλ ≈ 0.60 nm

# 连续 5 次观察到干涉条纹最不清晰（最差）时的 M1 读数 (mm，含 5 位小数估读)
# 示例数据 (相邻两次间距约为 0.289 mm 左右):
d_blur_readings = [
    Decimal("12.10503"),
    Decimal("12.39482"),
    Decimal("12.68435"),
    Decimal("12.97391"),
    Decimal("13.26354")
]

# 计算相邻两次模糊间距 Δd_k
step_diffs = []
for k in range(len(d_blur_readings) - 1):
    diff = d_blur_readings[k + 1] - d_blur_readings[k]
    step_diffs.append(diff)

avg_diff_mm = sum(step_diffs) / Decimal(str(len(step_diffs)))
avg_diff_nm = avg_diff_mm * Decimal("1000000")  # mm 转为 nm

# 计算波长差 Δλ = λ_bar^2 / (2 * Δd)
delta_lambda_calc = (LAMBDA_BAR_NM ** 2) / (Decimal("2") * avg_diff_nm)
rel_err_doublet = (abs(delta_lambda_calc - DELTA_LAMBDA_THEORY_NM) / DELTA_LAMBDA_THEORY_NM) * Decimal("100")

print("=" * 50)
print("           钠 光 D 双 线 波 长 差 测 定 结 果           ")
print("=" * 50)
for idx, diff in enumerate(step_diffs):
    print(f"第 {idx+1} 次到第 {idx+2} 次最模糊间距 : {diff:.5f} mm")
print("-" * 50)
print(f"条纹周期平均位移间隔 Δd_avg  : {avg_diff_mm:.5f} mm")
print(f"钠黄光平均波长 λ_bar          : {LAMBDA_BAR_NM} nm")
print(f"测得双线波长差 Δλ             : {delta_lambda_calc:.4f} nm")
print(f"理论参考波长差 Δλ_0           : {DELTA_LAMBDA_THEORY_NM} nm")
print(f"波长差相对误差               : {rel_err_doublet:.2f}%")
print("=" * 50)